In [2]:
import os
os.environ["MODEL_KEY"]="ollama"
from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, ModelSettings, set_tracing_disabled
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime

set_tracing_disabled(True)

In [3]:
local_model = OpenAIChatCompletionsModel(
    model="qwen3.5:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    )
)

In [4]:
bad_agent = Agent(
    model=local_model,
    name="Bad Agent",
    instructions="You are helpful."
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    model = local_model,

    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

Score: 5/10

Issues:
- Missing type hints reduces IDE support and clarity.
- No docstring explains expected inputs or outputs.
- No error handling for incompatible types (e.g., `None` or mixed types).

Suggestions:
- Add type annotations for better static analysis and readability.
- Include a docstring following PEP 257 standards.
- Validate inputs if strict numeric addition is required in your context.

```python
def add(x: int | float, y: int | float) -> int | float:
    """
    Add two numbers together.

    Args:
        x: First number.
        y: Second number.

    Returns:
        The sum of x and y.
    """
    return x + y
```


In [5]:


# Creative agent — high temperature
creative_agent = Agent(
    model=local_model,
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    )
)

# Precise agent — low temperature
precise_agent = Agent(
    name="Fact Checker",
    model=local_model,
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    )
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

 CREATIVE:
 

 PRECISE:
 


In [6]:
from pydantic import BaseModel, Field
from agents import Agent, Runner

class PodcastReview(BaseModel):
    title: str
    host: str
    rating: float
    genre: str
    technical_level: str
    summary: str
    recommendation: bool


reviewer = Agent(
    name="AI Podcast Reviewer",
    model=local_model,
    instructions="""
Return ONLY valid JSON.

No markdown. No explanation.

Format:
{
  "title": "...",
  "host": "...",
  "rating": 1-10,
  "genre": "...",
  "technical_level": "Beginner/Intermediate/Advanced",
  "summary": "...",
  "recommendation": true/false
}
""",
    output_type=PodcastReview,
)

result = await Runner.run(
    reviewer,
    "Review the podcast 'Lex Fridman Podcast'"
)

# ✅ SAFE DEBUG (always works)
print("FULL RESULT:")
print(result)

# ✅ Extract structured output
review = result.final_output

print("\nPARSED OUTPUT:")
print(review)

print("\nPARSED OUTPUT:")
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

FULL RESULT:
RunResult:
- Last agent: Agent(name="AI Podcast Reviewer", ...)
- Final output (PodcastReview):
    {
      "title": "Lex Fridman Podcast",
      "host": "Lex Fridman",
      "rating": 8.0,
      "genre": "Technology, Science, Philosophy",
      "technical_level": "Intermediate",
      "summary": "Long-form conversations exploring AI, robotics, history, and philosophy with leading experts, focusing on the nature of intelligence and humanity.",
      "recommendation": true
    }
- 1 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)

PARSED OUTPUT:
title='Lex Fridman Podcast' host='Lex Fridman' rating=8.0 genre='Technology, Science, Philosophy' technical_level='Intermediate' summary='Long-form conversations exploring AI, robotics, history, and philosophy with leading experts, focusing on the nature of intelligence and humanity.' recommendation=True

PARSED OUTPUT:
Title:          Lex Fridman Podcas

In [7]:
from typing import Literal
from pydantic import BaseModel, Field
from agents import Agent, Runner
from agents.model_settings import ModelSettings

# Define structured output
class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence score 0.0-1.0"
    )


# Create agent with strict JSON instructions
classifier = Agent(
    name="Product Classifier",
    model=local_model,
    instructions="""
You are a JSON API for product classification.

Analyze the customer's request and return ONLY valid JSON with EXACT keys:

{
  "category": "electronics|clothing|food|books|other",
  "urgency": "low|medium|high",
  "price_range": "budget|mid|premium",
  "search_query": "optimized search query string",
  "confidence": 0.0-1.0
}

Do NOT include markdown, bullets, or explanations. Return parsable JSON ONLY.
""",
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1)
)

# Test requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")


Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: medium | Price: budget
   Search: 'affordable student laptop for school' | Confidence: 90%

Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: medium | Price: mid
   Search: 'popular novels for birthday gift' | Confidence: 85%

Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: mid
   Search: 'phone charger' | Confidence: 95%


In [8]:
def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    name="Smart Scheduler",
    model=local_model,
    instructions=dynamic_instructions  # <-- Pass function, not string!
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

It's currently 2:50 PM on Thursday, April 9th, 2026.

Since it's the afternoon, this is the ideal time for **meetings and collaborations**. Here are a few suggestions:

*   Schedule a sync with your team or colleagues.
*   Work on collaborative projects or group tasks.
*   Respond to shared documents or communication threads.
*   Plan upcoming joint initiatives.

Would you like help scheduling a specific meeting?


In [15]:
from typing import Literal
from pydantic import BaseModel
import json

# ---- Schema ----
class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"]
    department: Literal["billing", "technical", "sales", "general"]
    summary: str
    suggested_response: str


# ---- Normalizers ----
def normalize_sentiment(v: str) -> str:
    v = v.lower()
    if "angry" in v or "urgent" in v:
        return "angry"
    if "neg" in v:
        return "negative"
    if "pos" in v:
        return "positive"
    return "neutral"


def normalize_priority(v: str) -> str:
    v = v.lower()
    if "p0" in v:
        return "P0-critical"
    if "p1" in v:
        return "P1-high"
    if "p2" in v:
        return "P2-medium"
    return "P3-low"


def normalize_department(v: str) -> str:
    v = v.lower()
    if "tech" in v:
        return "technical"
    if "bill" in v:
        return "billing"
    if "sale" in v:
        return "sales"
    return "general"


# ---- Agent (NO output_type) ----
ticket_analyzer = Agent(
    name="Ticket Analyzer",
    model=local_model,
    instructions="""
Return ONLY JSON:
{
  "sentiment": "...",
  "priority": "...",
  "department": "...",
  "summary": "...",
  "suggested_response": "..."
}
""",
    model_settings=ModelSettings(temperature=0.2)
)

# ---- Tickets ----
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

# ---- Runner ----
async def main():
    for ticket in tickets:
        result = await Runner.run(ticket_analyzer, ticket)

        # ✅ FIX HERE
        raw = result.final_output  

        # If model returns dict already
        if isinstance(raw, dict):
            data = raw
        else:
            try:
                data = json.loads(raw)
            except:
                print("❌ Invalid JSON:", raw)
                continue

        # Normalize
        data["sentiment"] = normalize_sentiment(data.get("sentiment", ""))
        data["priority"] = normalize_priority(data.get("priority", ""))
        data["department"] = normalize_department(data.get("department", ""))

        # Validate
        try:
            t = TicketAnalysis(**data)
        except Exception as e:
            print("❌ Validation failed:", e)
            continue

        print("\n" + "="*60)
        print("Ticket:", ticket[:60])
        print("Sentiment:", t.sentiment)
        print("Priority:", t.priority)
        print("Department:", t.department)
        print("Summary:", t.summary)
        print("Response:", t.suggested_response[:120])


# ---- Run ----
await main()


Ticket: Your API has been returning 500 errors for 2 hours. Our prod
Sentiment: negative
Priority: P3-low
Department: general
Summary: Customer reporting API 500 errors for 2 hours resulting in production downtime.
Response: We understand the severity of this issue and apologize for the impact on your production. Our engineering team is active

Ticket: Hi, I was charged twice this month. Can you please look into
Sentiment: neutral
Priority: P3-low
Department: billing
Summary: Customer reports being charged twice this month.
Response: Hello, I apologize for the inconvenience. I will investigate this duplicate charge immediately. Could you please share t

Ticket: Love the product! Any chance you'll add dark mode soon?
Sentiment: positive
Priority: P3-low
Department: general
Summary: User praises product and requests dark mode feature.
Response: Thank you for the wonderful feedback! We're so glad you're enjoying the product. Dark mode is a popular request, and I'v
